In [3]:
import requests
# Search for properties related to a keyword on the Wikidata SPARQL endpoint
# Used to manually determine which Wikidata property best matches our custom predicate
def search_wikidata_property(keyword):
    url = "https://query.wikidata.org/sparql"
    query = f"""
    SELECT ?property ?propertyLabel WHERE {{
        ?property a wikibase:Property .
        ?property rdfs:label ?propertyLabel .
        FILTER(CONTAINS(LCASE(?propertyLabel), "{keyword}"))
        FILTER(LANG(?propertyLabel) = "en")
    }}
    LIMIT 10
    """
    headers = {
        "User-Agent": "MovieKG-StudentProject/1.0 (han.wang@edu.devinci.fr)",
        "Accept": "application/json"
    }
    r = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=15)
    results = r.json()["results"]["bindings"]
    for row in results:
        print(row["property"]["value"], "|", row["propertyLabel"]["value"])

In [2]:
# Search by keyword and manually determine the most appropriate match
print("Searching Wikidata properties...\n")
for keyword in ["director", "cast member", "genre", "production company", "publication date"]:
    print(f"--- {keyword} ---")
    search_wikidata_property(keyword)
    print()


Searching Wikidata properties...

--- director ---
http://www.wikidata.org/entity/P633 | Quebec cultural heritage directory ID
http://www.wikidata.org/entity/P57 | director
http://www.wikidata.org/entity/P344 | director of photography
http://www.wikidata.org/entity/P4227 | Indonesian Small Islands Directory ID
http://www.wikidata.org/entity/P4283 | LUMIERE director ID
http://www.wikidata.org/entity/P3174 | art director
http://www.wikidata.org/entity/P5115 | Directory of Open Access Journals ID
http://www.wikidata.org/entity/P4345 | director of archaeological fieldwork
http://www.wikidata.org/entity/P5126 | assistant director
http://www.wikidata.org/entity/P1037 | director / manager

--- cast member ---
http://www.wikidata.org/entity/P161 | cast member

--- genre ---
http://www.wikidata.org/entity/P136 | genre
http://www.wikidata.org/entity/P4953 | Library of Congress Genre/Form Terms ID
http://www.wikidata.org/entity/P6352 | OLAC video game genre vocabulary ID
http://www.wikidata.org/e

In [4]:
from rdflib import Graph, Namespace, URIRef, OWL, RDFS

MVO = Namespace("http://moviekg.local/ontology/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

pred_g = Graph()
pred_g.bind("mvo", MVO)
pred_g.bind("wdt", WDT)
pred_g.bind("owl", OWL)

# Based on the results of the SPARQL query, manually select the most relevant Wikidata property:
alignments = [
    ("hasDirector", "P57"),
    ("hasActor",    "P161"),
    ("hasGenre",    "P136"),
    ("producedBy",  "P272"),
    ("releaseYear", "P577"),
]

# owl:equivalentProperty indicates that two properties are semantically equivalent
for local_prop, wikidata_prop in alignments:
    pred_g.add((
        MVO[local_prop],
        OWL.equivalentProperty,
        WDT[wikidata_prop]
    ))

# hasTitle has no equivalent property, marked as subPropertyOf rdfs:label
pred_g.add((
    MVO.hasTitle,
    RDFS.subPropertyOf,
    RDFS.label
))

pred_g.serialize("predicate_alignment.ttl", format="turtle")

<Graph identifier=N905faeaddbcd44f3a0576988f98243ae (<class 'rdflib.graph.Graph'>)>